In [1]:
!pip install faker pandas pyarrow duckdb

In [2]:
import pandas as pd
import duckdb

def run_transformations(input_path="/home/jovyan/work/data", db_path="/home/jovyan/work/data/gas_warehouse.duckdb"):
    # Beolvasás
    df_readings = pd.read_parquet(f"{input_path}/readings.parquet")
    df_devices = pd.read_csv(f"{input_path}/devices.csv")

    # 1. Adattisztítás: Szűrés és hibás adatok eltávolítása
    df_readings = df_readings.dropna(subset=['consumption'])
    df_readings['timestamp'] = pd.to_datetime(df_readings['timestamp'])

    # 2. Adattranszformáció: Dim_Time létrehozása
    df_time = pd.DataFrame()
    df_time['timestamp'] = df_readings['timestamp'].unique()
    df_time['hour'] = df_time['timestamp'].dt.hour
    df_time['day'] = df_time['timestamp'].dt.day
    df_time['month'] = df_time['timestamp'].dt.month
    df_time['is_weekend'] = df_time['timestamp'].dt.dayofweek > 4

    # 3. Aggregáció
    df_daily_agg = df_readings.groupby([df_readings['timestamp'].dt.date, 'device_id'])['consumption'].sum().reset_index()
    df_daily_agg.columns = ['date', 'device_id', 'daily_total_consumption']

    # 4. Betöltés DuckDB-be
    con = duckdb.connect(db_path)

    # 5. Tábla létrehozása és adatok betöltése
    con.execute("CREATE OR REPLACE TABLE dim_device AS SELECT * FROM df_devices")
    con.execute("CREATE OR REPLACE TABLE dim_time AS SELECT * FROM df_time")
    con.execute("CREATE OR REPLACE TABLE fact_gas_usage AS SELECT * FROM df_readings")
    con.execute("CREATE OR REPLACE TABLE agg_daily_usage AS SELECT * FROM df_daily_agg") # Aggregált tábla
    
    con.close()
    print(f"Transformation and loading to DuckDB finished.")